# LP-Prop v2 · Notebook 1: làm sạch dữ liệu và sinh lại câu hỏi

**Input:** `lp-prop-work-d1`, `lp-prop-pipeline-s2`. **Accelerator:** GPU T4 ×2. **Internet:** On.

1. **s1_clean** (CPU, khoảng 1 phút): bỏ văn bản dự thảo, bỏ Điều chỉ có tiêu đề, gộp Điều trùng, sửa passage bị sót, lọc props s2 (không chạy lại LLM của s2), tạo bản đồ embedding cũ.
2. **s1b** (GPU, Qwen2.5-14B-AWQ trên 2 GPU): sinh câu hỏi cho tối đa 2.400 gold span ở tập eval, kiểm tra tự động, chọn **1.200 câu**.

Thời gian ước tính (chưa đo): **khoảng 1–1,5 giờ**. Chạy bằng **Save Version → Save & Run All**.
Output (`/kaggle/working/data_v2`, `/kaggle/working/questions_v2`) là **input của Notebook 2**.

In [ ]:
!pip install -q vllm
# vLLM nâng torch lên CUDA 13; torchaudio cũ làm lỗi import -> gỡ (không cần)
!pip uninstall -y -q torchaudio
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/s1_clean.py
"""
s1_clean.py — LP-Prop v2: làm sạch corpus từ dữ liệu đã có (D1 + S2/work_clean). Chỉ dùng CPU.

Việc làm:
  1. Bỏ Điều chỉ có tiêu đề (mục lục bị cào nhầm, body < 5 từ sau khi bỏ dòng CHƯƠNG/MỤC/số trang).
  2. Bỏ toàn bộ văn bản dự thảo (doc_name bắt đầu bằng "Dự thảo").
  3. Gộp Điều trùng nội dung: giữ aid nhỏ nhất trong mỗi nhóm.
  4. Passage: giữ passage cũ của các Điều còn lại; tạo passage cho các Điều bị sót
     (bug split_khoan cắt rỗng Điều một đoạn / Điều kết thúc bằng dòng CHƯƠNG ...).
  5. Lọc gold_spans, qa_generic, props_* (s2) theo Điều còn lại. KHÔNG chạy lại LLM của s2:
     proposition được tách riêng cho từng Điều nên Điều giữ lại có nội dung y hệt.
  6. Ghi emb_map.json: với mỗi tập đơn vị (passage, std, lp, ...) — vị trí của từng đơn vị mới trong
     file embedding cũ của s3 (emb_<set>_bge-m3.npy), -1 nếu là đơn vị mới -> s3_v2 không phải embed lại.

Dùng:
  python s1_clean.py --input_root /kaggle/input --out /kaggle/working/data_v2
"""
import argparse, glob, hashlib, importlib.util, json, os, re
from collections import Counter, OrderedDict, defaultdict

PROP_SETS = ["std", "lp", "std_raw", "lp_raw", "lp_edge_only"]
HEAD_LINE = re.compile(r"^\s*(CHƯƠNG|Chương|MỤC|Mục|PHẦN|Phần)\s+[IVXLCDM\d]+", re.U)
PAGE_NUM = re.compile(r"\.{0,3}\s*\d{1,3}\s*$")


def read_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                try:
                    out.append(json.loads(l))
                except json.JSONDecodeError:
                    pass
    return out


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def find(root, name, prefer=None):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
    if prefer:
        hits = [h for h in hits if prefer in h] or hits
    if not hits:
        raise FileNotFoundError(f"không thấy {name} dưới {root}")
    return hits[0]


def real_body(article):
    """Phần thân Điều sau khi bỏ dòng tiêu đề Điều, dòng CHƯƠNG/MỤC và số trang mục lục."""
    lines = article.split("\n")[1:]
    keep = []
    for l in lines:
        s = l.strip()
        if not s or HEAD_LINE.match(s):
            continue
        if s.isupper() and len(s.split()) <= 20:      # dòng tên chương viết hoa
            continue
        keep.append(s)
    body = " ".join(keep)
    return PAGE_NUM.sub("", body).strip() if len(body.split()) < 30 else body


def make_passages(aid, article, split_khoan, target=(120, 190)):
    """Tạo passage cho Điều bị sót. Ưu tiên split_khoan của s2; nếu nó cắt rỗng thì dùng real_body."""
    pieces = [t.strip() for t in split_khoan(article).values() if t and t.strip()]
    if not pieces or sum(len(p.split()) for p in pieces) < 0.5 * len(real_body(article).split()):
        pieces = [p for p in re.split(r"\n\s*\n|\n", "\n".join(article.split("\n")[1:])) if p.strip()]
        pieces = [re.sub(r"^\s*\d+\.\s*", "", p).strip() for p in pieces
                  if not HEAD_LINE.match(p.strip()) and not (p.strip().isupper() and len(p.split()) <= 20)]
    # tách đoạn quá dài theo câu
    sents = []
    for p in pieces:
        if len(p.split()) <= target[1]:
            sents.append(p)
        else:
            sents += [s for s in re.split(r"(?<=[.;:])\s+", p) if s.strip()]
    out, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur).split()) + len(s.split()) > target[1] and len(" ".join(cur).split()) >= target[0] // 2:
            out.append(" ".join(cur)); cur = []
        cur.append(s)
    if cur:
        out.append(" ".join(cur))
    return [{"uid": f"{aid}:c{i}", "aid": aid, "text": t} for i, t in enumerate(out) if t.split()]


def ordered_units(records):
    """Thứ tự đơn vị giống build_pointers() của s3 cũ: vị trí = lần xuất hiện đầu, nội dung = lần cuối."""
    units = OrderedDict()
    for r in records:
        for p in r["props"]:
            units[p["uid"]] = p
    return units


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_root", default="/kaggle/input")
    ap.add_argument("--out", default="/kaggle/working/data_v2")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)

    p_articles = find(a.input_root, "articles_all.jsonl")
    p_gold = find(a.input_root, "gold_spans.jsonl", prefer="work-d1")
    p_pas = find(a.input_root, "passages.jsonl", prefer="work-d1")
    p_gen = find(a.input_root, "qa_generic.jsonl", prefer="work-d1")
    p_s2w = find(a.input_root, "s2_worker.py")
    p_props = {m: find(a.input_root, f"props_{m}.jsonl", prefer="work_clean") for m in PROP_SETS}
    print("[input]", json.dumps({"articles": p_articles, "gold": p_gold, "passages": p_pas, "generic": p_gen,
                                 "s2_worker": p_s2w, **{f"props_{k}": v for k, v in p_props.items()}}, ensure_ascii=False, indent=1))

    spec = importlib.util.spec_from_file_location("s2w", p_s2w); s2w = importlib.util.module_from_spec(spec); spec.loader.exec_module(s2w)

    arts = read_jsonl(p_articles)
    reason = {}
    for x in arts:
        if x["doc_name"].strip().startswith("Dự thảo"):
            reason[x["aid"]] = "draft"
        elif len(real_body(x["article_content"]).split()) < 5:
            reason[x["aid"]] = "heading_only"
    groups = defaultdict(list)
    for x in arts:
        if x["aid"] not in reason:
            groups[hashlib.md5(x["article_content"].strip().encode("utf-8")).hexdigest()].append(x["aid"])
    dup_of = {}
    for v in groups.values():
        if len(v) > 1:
            keep = min(v, key=lambda s: (len(s), s))
            for aid in v:
                if aid != keep:
                    reason[aid] = "duplicate"; dup_of[aid] = keep
    kept = [x for x in arts if x["aid"] not in reason]
    K = {x["aid"] for x in kept}
    write_jsonl(os.path.join(a.out, "articles_v2.jsonl"), kept)

    # passages
    old_pas = read_jsonl(p_pas)
    old_pas_row = {p["uid"]: i for i, p in enumerate(old_pas)}
    pas = [p for p in old_pas if p["aid"] in K]
    have = {p["aid"] for p in pas}
    new_pas = []
    for x in kept:
        if x["aid"] not in have:
            new_pas += make_passages(x["aid"], x["article_content"], s2w.split_khoan)
    rows = [old_pas_row[p["uid"]] for p in pas] + [-1] * len(new_pas)
    pas += new_pas
    write_jsonl(os.path.join(a.out, "passages.jsonl"), pas)
    emb_map = {"passage": {"n_old": len(old_pas), "rows": rows}}

    # gold spans, generic QA
    gold = [g for g in read_jsonl(p_gold) if g["aid"] in K]
    write_jsonl(os.path.join(a.out, "gold_spans.jsonl"), gold)
    gen = [q for q in read_jsonl(p_gen) if q["aid"] in K]
    write_jsonl(os.path.join(a.out, "qa_generic.jsonl"), gen)

    # props của s2 + bản đồ embedding cũ
    prop_stats = {}
    for m, path in p_props.items():
        old = read_jsonl(path)
        old_units = ordered_units(old)
        old_row = {u: i for i, u in enumerate(old_units)}
        new = []
        for r in old:
            if r["aid"] not in K:
                continue
            seen = OrderedDict()
            for p in r["props"]:
                seen[p["uid"]] = old_units[p["uid"]]          # nội dung = lần xuất hiện cuối (giống s3 cũ)
            valid = set(seen)
            ptrs = [e for e in r.get("pointers", []) if e["head"] in valid and e["tail"] in valid]
            new.append(dict(r, props=list(seen.values()), pointers=ptrs))
        write_jsonl(os.path.join(a.out, f"props_{m}.jsonl"), new)
        units = ordered_units(new)
        emb_map[m] = {"n_old": len(old_units), "rows": [old_row[u] for u in units]}
        prop_stats[m] = {"articles": len(new), "props": sum(len(r["props"]) for r in new),
                         "pointers": sum(len(r.get("pointers", [])) for r in new)}
    json.dump(emb_map, open(os.path.join(a.out, "emb_map.json"), "w"))

    ev = [x for x in kept if x.get("split") == "eval"]
    stats = {
        "articles_in": len(arts), "articles_kept": len(kept), "articles_eval": len(ev),
        "docs_kept": len({x["doc_name"] for x in kept}),
        "removed": dict(Counter(reason.values())), "duplicate_of": dup_of,
        "passages": len(pas), "passages_new": len(new_pas), "articles_with_new_passages": len({p["aid"] for p in new_pas}),
        "gold_spans": len(gold), "gold_spans_eval": sum(g["aid"] in {x["aid"] for x in ev} for g in gold),
        "qa_generic": len(gen), "props": prop_stats,
    }
    json.dump(stats, open(os.path.join(a.out, "stats_v2.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump(reason, open(os.path.join(a.out, "removed_aids.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=0)
    print(json.dumps({k: v for k, v in stats.items() if k != "duplicate_of"}, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/s1b_gen_questions.py
"""
s1b_gen_questions.py — LP-Prop v2: sinh lại câu hỏi có điều kiện, mỗi câu gắn với MỘT gold span
(quy tắc + điều kiện/ngoại lệ), sao cho muốn trả lời đúng thì bắt buộc phải nêu điều kiện.

Giai đoạn:
  gen    : sinh câu hỏi bằng LLM (vLLM), có checkpoint, chạy lại sẽ làm tiếp phần còn thiếu
  select : kiểm tra tự động + chọn đúng --target câu (mặc định 1.200) cho tập eval

Kiểm tra tự động (mọi câu phải qua hết):
  json_ok          : đọc được JSON, đủ 3 trường
  q_form           : câu hỏi 8–90 từ, kết thúc bằng '?'
  cond_in_answer   : đáp án chứa điều kiện (>=50% cụm 4 từ, hoặc >=70% từ)
  rule_in_answer   : đáp án chứa quy tắc (>=40% từ)
  no_copy          : câu hỏi KHÔNG chép nguyên văn điều kiện (<80% cụm 4 từ)
  ignored_differs  : đáp án "bỏ qua điều kiện" không chứa điều kiện và khác đáp án đúng
  short_answer     : đáp án <= 130 từ

Dùng:
  python s1b_gen_questions.py --stage gen    --data /kaggle/working/data_v2 --out /kaggle/working/questions_v2
  python s1b_gen_questions.py --stage select --data /kaggle/working/data_v2 --out /kaggle/working/questions_v2
"""
import argparse, json, os, random, re, time, unicodedata
from collections import Counter, defaultdict

DEFAULT_MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
STOP = set("của và các là được có cho theo với trong này đó những một người phải thì".split())


def norm(s):
    return unicodedata.normalize("NFC", s or "")

def toks(s):
    return [w for w in re.findall(r"\w+", norm(s).lower()) if w not in STOP]

def coverage(span, texts):
    st = toks(span)
    if not st:
        return 0.0
    bag = set(w for t in texts for w in toks(t))
    return sum(1 for w in st if w in bag) / len(st)

def phrase_frac(span, text, n=4):
    c, a = toks(span), toks(text)
    if len(c) < n:
        return 1.0 if c and " ".join(c) in " ".join(a) else 0.0
    ag = {tuple(a[i:i + n]) for i in range(len(a) - n + 1)}
    cg = [tuple(c[i:i + n]) for i in range(len(c) - n + 1)]
    return sum(g in ag for g in cg) / len(cg)

def read_jsonl(path):
    out = []
    if not os.path.exists(path):
        return out
    with open(path, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                try:
                    out.append(json.loads(l))
                except json.JSONDecodeError:
                    pass
    return out

def write_jsonl(path, rows, mode="w"):
    with open(path, mode, encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


# ---------------- prompt ----------------
SYS = ("Bạn là chuyên gia pháp luật Việt Nam, soạn câu hỏi kiểm tra việc hiểu ĐIỀU KIỆN áp dụng của quy định pháp luật. "
       "Chỉ trả về đúng một đối tượng JSON, không giải thích.")

EXAMPLE = """VÍ DỤ
VĂN BẢN: Bộ luật Lao động (ví dụ minh hoạ)
QUY TẮC: Người lao động được nghỉ hằng năm 12 ngày làm việc
ĐIỀU KIỆN/NGOẠI LỆ: nếu làm việc đủ 12 tháng cho một người sử dụng lao động
JSON:
{"question": "Theo Bộ luật Lao động, người lao động mới làm việc 8 tháng cho một công ty có được nghỉ hằng năm 12 ngày làm việc không?",
 "answer": "Chưa được hưởng đủ 12 ngày. Theo Bộ luật Lao động, người lao động chỉ được nghỉ hằng năm 12 ngày làm việc nếu đã làm việc đủ 12 tháng cho một người sử dụng lao động.",
 "answer_if_condition_ignored": "Có, người lao động được nghỉ hằng năm 12 ngày làm việc."}
"""

def article_window(text, head, max_words=900):
    w = text.split()
    if len(w) <= max_words:
        return text
    key = " ".join(head.split()[:6])
    pos = text.find(key)
    start_w = len(text[:pos].split()) if pos >= 0 else 0
    s = max(0, start_w - max_words // 3)
    return ("… " if s > 0 else "") + " ".join(w[s:s + max_words]) + " …"

def gen_messages(c):
    user = (
        f"{EXAMPLE}\nBÂY GIỜ LÀM VỚI QUY ĐỊNH SAU\n"
        f"VĂN BẢN: {c['doc_name']}\n"
        f"ĐIỀU LUẬT:\n{article_window(c['article'], c['head'])}\n\n"
        f"QUY TẮC: {c['head']}\n"
        f"ĐIỀU KIỆN/NGOẠI LỆ: {c['condition']}\n\n"
        "Viết MỘT câu hỏi tiếng Việt sao cho muốn trả lời đúng thì BẮT BUỘC phải nêu điều kiện/ngoại lệ trên.\n"
        "Yêu cầu:\n"
        "- Câu hỏi nêu tên văn bản luật để biết đang hỏi luật nào (không cần số Điều).\n"
        "- Câu hỏi hỏi về quy tắc (ai phải/được làm gì, thời hạn, thẩm quyền, trường hợp cụ thể...). "
        "KHÔNG chép nguyên văn điều kiện vào câu hỏi; có thể nêu một tình huống cụ thể mà điều kiện quyết định câu trả lời.\n"
        "- Trả lời mà bỏ qua điều kiện/ngoại lệ thì sẽ sai hoặc khái quát quá mức.\n"
        "- answer: 1–3 câu, nêu quy tắc VÀ nêu rõ điều kiện/ngoại lệ, chỉ dựa vào Điều luật trên.\n"
        "- answer_if_condition_ignored: câu trả lời nghe hợp lý nhưng bỏ mất điều kiện/ngoại lệ.\n"
        'Chỉ trả về JSON: {"question": "...", "answer": "...", "answer_if_condition_ignored": "..."}'
    )
    return [{"role": "system", "content": SYS}, {"role": "user", "content": user}]

def parse_json(raw):
    t = (raw or "").strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    try:
        js = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    if not all(isinstance(js.get(k), str) and js[k].strip() for k in ("question", "answer", "answer_if_condition_ignored")):
        return None
    return {k: js[k].strip() for k in ("question", "answer", "answer_if_condition_ignored")}

def checks(c, js):
    if js is None:
        return {"json_ok": False}
    q, ans, ign = js["question"], js["answer"], js["answer_if_condition_ignored"]
    return {
        "json_ok": True,
        "q_form": 8 <= len(q.split()) <= 90 and q.rstrip().endswith("?"),
        "cond_in_answer": phrase_frac(c["condition"], ans) >= 0.5 or coverage(c["condition"], [ans]) >= 0.7,
        "rule_in_answer": coverage(c["head"], [ans]) >= 0.4,
        "no_copy": phrase_frac(c["condition"], q) < 0.8,
        "ignored_differs": phrase_frac(c["condition"], ign) < 0.5 and norm(ign).strip() != norm(ans).strip(),
        "short_answer": len(ans.split()) <= 130,
    }


# ---------------- generator ----------------
class VLLMGenerator:
    def __init__(self, model, tp=1, max_model_len=6144, gpu_util=0.90):
        from vllm import LLM
        print(f"[vllm] nạp {model} (tensor_parallel={tp})", flush=True)
        self.llm = LLM(model, dtype="half", max_model_len=max_model_len, tensor_parallel_size=tp,
                       gpu_memory_utilization=gpu_util, seed=42)

    def generate(self, msgs, max_tokens=512):
        from vllm import SamplingParams
        sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=42)
        return [o.outputs[0].text for o in self.llm.chat(msgs, sp, use_tqdm=True)]

class MockGenerator:
    def generate(self, msgs, max_tokens=512):
        out = []
        for m in msgs:
            u = m[1]["content"].split("BÂY GIỜ LÀM VỚI QUY ĐỊNH SAU")[1]
            head = re.search(r"QUY TẮC: (.*)", u).group(1); cond = re.search(r"ĐIỀU KIỆN/NGOẠI LỆ: (.*)", u).group(1)
            doc = re.search(r"VĂN BẢN: (.*)", u).group(1)
            out.append(json.dumps({"question": f"Theo {doc}, {head.split(',')[0].lower()} trong những trường hợp nào?",
                                   "answer": f"{head}, {cond}.", "answer_if_condition_ignored": f"{head}."}, ensure_ascii=False))
        return out


# ---------------- ứng viên ----------------
def build_candidates(data, max_candidates, seed):
    arts = {x["aid"]: x for x in read_jsonl(os.path.join(data, "articles_v2.jsonl"))}
    spans = [g for g in read_jsonl(os.path.join(data, "gold_spans.jsonl"))
             if arts.get(g["aid"], {}).get("split") == "eval" and g.get("condition") and g.get("head")]
    by_a = defaultdict(list)
    for g in spans:
        by_a[g["aid"]].append(g)
    rng = random.Random(seed)
    aids = sorted(by_a); rng.shuffle(aids)
    for a in aids:
        rng.shuffle(by_a[a])
    cands, rnd = [], 0
    while len(cands) < max_candidates and any(len(by_a[a]) > rnd for a in aids):
        for a in aids:
            if len(by_a[a]) > rnd and len(cands) < max_candidates:
                g = by_a[a][rnd]; x = arts[a]
                cands.append({"sid": g["sid"], "aid": a, "round": rnd, "type": g["type"], "cue": g.get("cue", ""),
                              "head": g["head"], "condition": g["condition"], "khoan": g.get("khoan"),
                              "doc_name": x["doc_name"], "article": x["article_content"], "has_exc": x.get("has_exc")})
        rnd += 1
    return cands


def stage_gen(a):
    cands = build_candidates(a.data, a.max_candidates, a.seed)
    out = os.path.join(a.out, "gen_raw.jsonl")
    done = {r["sid"] for r in read_jsonl(out) if parse_json(r["raw"]) is not None}
    todo = [c for c in cands if c["sid"] not in done]
    print(f"[gen] {len(cands)} ứng viên, còn {len(todo)} cần sinh -> {out}", flush=True)
    if not todo:
        return True
    gen = MockGenerator() if a.mock else VLLMGenerator(a.model, a.tp)
    deadline = time.time() + a.deadline_min * 60 if a.deadline_min else None
    t0 = time.time()
    for i in range(0, len(todo), a.batch):
        if deadline and time.time() > deadline:
            print("[checkpoint] chạm deadline, dừng để lưu"); return False
        chunk = todo[i:i + a.batch]
        raws = gen.generate([gen_messages(c) for c in chunk])
        write_jsonl(out, [{"sid": c["sid"], "raw": r} for c, r in zip(chunk, raws)], mode="a")
        n = i + len(chunk)
        print(f"[gen] {n}/{len(todo)} — {(time.time() - t0) / n:.2f} s/câu", flush=True)
    return True


def stage_select(a):
    cands = {c["sid"]: c for c in build_candidates(a.data, a.max_candidates, a.seed)}
    raw = {}
    for r in read_jsonl(os.path.join(a.out, "gen_raw.jsonl")):
        if r["sid"] in cands and (r["sid"] not in raw or parse_json(raw[r["sid"]]) is None):
            raw[r["sid"]] = r["raw"]
    rows, fail = [], Counter()
    for sid, c in cands.items():
        if sid not in raw:
            fail["not_generated"] += 1; continue
        js = parse_json(raw[sid]); ck = checks(c, js)
        bad = [k for k, v in ck.items() if not v]
        for k in bad:
            fail[k] += 1
        if bad:
            continue
        rows.append({"c": c, "js": js})
    # chọn: ưu tiên vòng 0 (mỗi Điều 1 câu), rồi mới tới vòng sau
    rng = random.Random(a.seed)
    rows.sort(key=lambda r: r["c"]["sid"]); rng.shuffle(rows)
    rows.sort(key=lambda r: r["c"]["round"])
    chosen = rows[:a.target]
    per_aid = Counter()
    out = []
    for r in rows:
        c, js = r["c"], r["js"]
        k = per_aid[c["aid"]]; per_aid[c["aid"]] += 1
        out.append({"qid": f"{c['aid']}:c{k}", "aid": c["aid"], "sid": c["sid"], "split": "eval", "has_exc": c["has_exc"],
                    "question": js["question"], "answer": js["answer"], "answer_if_condition_ignored": js["answer_if_condition_ignored"],
                    "difficulty": "n/a", "question_type": "conditional", "is_conditional": True,
                    "condition_type": c["type"], "condition_cue": c["cue"], "head": c["head"],
                    "condition": c["condition"], "condition_needed": c["condition"], "gen_round": c["round"]})
    chosen_sids = {r["c"]["sid"] for r in chosen}
    sel = [q for q in out if q["sid"] in chosen_sids]
    write_jsonl(os.path.join(a.out, "cond_questions_v2_all_valid.jsonl"), out)
    write_jsonl(os.path.join(a.out, f"cond_questions_v2_eval_{a.target}.jsonl"), sel)
    n_gen = len(cands) - fail["not_generated"]
    stats = {"candidates": len(cands), "generated": n_gen, "valid": len(out), "selected": len(sel),
             "pass_rate": round(len(out) / max(n_gen, 1), 4), "fail_counts": dict(fail),
             "selected_types": dict(Counter(q["condition_type"] for q in sel)),
             "selected_rounds": dict(Counter(q["gen_round"] for q in sel)),
             "selected_articles": len({q["aid"] for q in sel}), "target_met": len(sel) >= a.target}
    json.dump(stats, open(os.path.join(a.out, "gen_stats.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    # 20 câu mẫu để đọc nhanh
    L = ["# 20 câu hỏi mẫu (LP-Prop v2)", ""]
    for q in random.Random(1).sample(sel, min(20, len(sel))):
        L += [f"**{q['qid']}** ({q['condition_type']})", f"- Hỏi: {q['question']}", f"- Đáp án: {q['answer']}",
              f"- Bỏ điều kiện: {q['answer_if_condition_ignored']}", f"- Điều kiện gốc: _{q['condition']}_", ""]
    open(os.path.join(a.out, "sample_questions.md"), "w", encoding="utf-8").write("\n".join(L))
    print(json.dumps(stats, ensure_ascii=False, indent=1))
    if not stats["target_met"]:
        print(f"[!] chỉ có {len(sel)} câu hợp lệ < {a.target}: tăng --max_candidates rồi chạy lại --stage gen và select")
    return stats["target_met"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["gen", "select"], required=True)
    ap.add_argument("--data", default="/kaggle/working/data_v2")
    ap.add_argument("--out", default="/kaggle/working/questions_v2")
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--tp", type=int, default=2)
    ap.add_argument("--target", type=int, default=1200)
    ap.add_argument("--max_candidates", type=int, default=2400)
    ap.add_argument("--batch", type=int, default=256)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--deadline_min", type=float, default=0)
    ap.add_argument("--mock", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    ok = stage_gen(a) if a.stage == "gen" else stage_select(a)
    print("[done]" if ok else "[chưa xong]")


if __name__ == "__main__":
    main()


In [ ]:
import os, subprocess, json, glob, time
def run(cmd, log=None, env=None):
    """Chạy lệnh trong /kaggle/working; nếu có log thì in dòng cuối của log."""
    full = f"cd /kaggle/working && {cmd}" + (f" > {log} 2>&1" if log else "")
    rc = subprocess.run(full, shell=True, env={**os.environ, **(env or {})}).returncode
    if log:
        lines = open(f"/kaggle/working/{log}", encoding="utf-8", errors="ignore").read().strip().splitlines()
        print(f"[{log}] " + (lines[-1] if lines else "(trống)"))
    return rc
def log_done(log):
    p = f"/kaggle/working/{log}"
    return os.path.exists(p) and open(p, encoding="utf-8", errors="ignore").read().rstrip().endswith("[done]")

## Bước 1: làm sạch dữ liệu

In [ ]:
run("python s1_clean.py --input_root /kaggle/input --out /kaggle/working/data_v2", log="s1_clean.log")
print(open("/kaggle/working/data_v2/stats_v2.json", encoding="utf-8").read()[:1500])

## Bước 2: sinh câu hỏi và kiểm tra tự động

In [ ]:
GEN = "python s1b_gen_questions.py --data /kaggle/working/data_v2 --out /kaggle/working/questions_v2 --tp 2"
max_cand = 2400
for attempt in range(4):
    for _ in range(3):   # chạm deadline / lỗi giữa chừng -> chạy tiếp phần còn thiếu
        run(f"{GEN} --stage gen --max_candidates {max_cand} --deadline_min 600", log="gen.log")
        if log_done("gen.log"):
            break
    run(f"{GEN} --stage select --max_candidates {max_cand}", log="select.log")
    if log_done("select.log"):
        break
    max_cand += 800          # chưa đủ 1.200 câu hợp lệ -> thêm ứng viên rồi sinh tiếp
    print("chưa đủ câu hợp lệ, tăng ứng viên lên", max_cand)
print(open("/kaggle/working/questions_v2/gen_stats.json", encoding="utf-8").read())

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/questions_v2/sample_questions.md", encoding="utf-8").read()))

Xong. Kiểm tra `gen_stats.json` (`target_met: true`) và đọc qua 20 câu mẫu ở trên.
Sau đó mở **Notebook 2**, chọn *Add Input → Your Work → Notebooks* và thêm notebook này (bản đã Save Version).